In [5]:
"""
Monte Carlo season simulation for Metro League Boys Soccer (2025-26),
using:
  - your scraped fixtures from metroleague_results_all_games.csv (home/away preserved)
  - Elo ratings trained from historical played games (filtered to Metro teams)
  - optional inclusion of already-played 2025-26 games as “fixed results”
  - N simulations to estimate seed probabilities and expected points

Outputs (in OUTPUT_DIR):
  - metroleague_2025-26_sim_seed_probabilities.csv
  - metroleague_2025-26_sim_points_summary.csv
  - metroleague_2025-26_sim_settings.txt
"""

from __future__ import annotations

import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# =========================
# CONFIG
# =========================
INPUT_CSV = Path(r"C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\metroleague_results_all_games.csv")
OUTPUT_DIR = Path(r"C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\Monte Carlo Sim Output\2.24")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASON_TO_SIMULATE = "2025-26"
N_SIMS = 100000
SEED = 42

# Elo params (match your model)
BASE_ELO = 1500.0
K = 40.0
HFA = 100.0
CAP_MARGIN = 3
UPSET_MULTIPLIER = 1.5

# If True: simulate sequentially and update Elo after each simulated match (more realistic, slower)
UPDATE_ELO_DURING_SIM = True

# Soccer draw handling
# If None, we estimate draw rate from historical Metro-vs-Metro matches prior to SEASON_TO_SIMULATE
DRAW_RATE = None  # e.g. 0.18

# Exclude these game types if present
EXCLUDE_GAME_TYPES = {"Scrimmage", "Jamboree", "Preseason", "Pre-Season"}

# Metro teams to include (allowlist)
METRO_TEAMS_RAW = [
    "Lincoln (Seattle)",
    "Bishop Blanchet",
    "Seattle Prep",
    "Ballard",
    "Ingraham",
    "O'Dea",
    "Eastside Catholic",
    "Lakeside (Seattle)",
    "Garfield",
    "Seattle Academy",
    "Roosevelt",
    "West Seattle",
    "Franklin",
    "Nathan Hale",
    "Chief Sealth",
    "Rainier Beach",
    "Cleveland",
]

TEAM_NAME_CORRECTIONS = {
    "Lakeside": "Lakeside (Seattle)",
    "Lakeside (Sea)": "Lakeside (Seattle)",
    "Seattle Prep.": "Seattle Prep",
}

ORDINAL_SUFFIX_RE = re.compile(r"(\d+)(st|nd|rd|th)\b", re.IGNORECASE)


# =========================
# NORMALIZATION
# =========================
def normalize_team_name(name) -> str:
    if name is None:
        return ""
    if isinstance(name, float) and pd.isna(name):
        return ""
    s = str(name).strip()
    s = s.replace("’", "'").replace("‘", "'")
    s = re.sub(r"\s+", " ", s)
    return TEAM_NAME_CORRECTIONS.get(s, s)


def canonicalize(name: str) -> str:
    s = normalize_team_name(name).lower()
    s = re.sub(r"[^a-z0-9()' ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


METRO_TEAMS = {canonicalize(t) for t in METRO_TEAMS_RAW}


# =========================
# DATE PARSING
# =========================
def season_years(season: str) -> tuple[int, int]:
    a, b = season.split("-")
    start = int(a)
    end = int(b)
    if end < 100:
        end += 2000
    return start, end


def infer_year_from_season_and_month(season: str, month: int, rollover_month: int = 7) -> int:
    start, end = season_years(season)
    return start if month >= rollover_month else end


def _clean_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none"}:
        return ""
    return s


def parse_match_datetime(season: str, date_raw, time_raw) -> pd.Timestamp:
    d = _clean_text(date_raw)
    if not d:
        return pd.NaT

    d = ORDINAL_SUFFIX_RE.sub(r"\1", d)
    d = re.sub(r"^\w+,\s*", "", d)  # remove weekday prefix
    d = re.sub(r"\s+", " ", d)

    t = _clean_text(time_raw)
    if not t or t.upper() == "TBD":
        t = "12:00 PM"

    dt = pd.to_datetime(f"{d} {t}", errors="coerce")
    if pd.notna(dt) and dt.year > 1900:
        return dt

    d_only = pd.to_datetime(d, errors="coerce")
    if pd.isna(d_only):
        return pd.NaT

    inferred_year = infer_year_from_season_and_month(season, int(d_only.month), rollover_month=7)
    return pd.to_datetime(
        f"{inferred_year}-{int(d_only.month):02d}-{int(d_only.day):02d} {t}",
        errors="coerce",
    )


# =========================
# ELO CORE
# =========================
def expected_home_win_prob(home_elo: float, away_elo: float, hfa: float) -> float:
    return 1.0 / (1.0 + 10 ** ((away_elo - (home_elo + hfa)) / 400.0))


def elo_update(home_elo, away_elo, result_home, k=K, hfa=HFA, cap_margin=CAP_MARGIN, upset_multiplier=UPSET_MULTIPLIER, margin=1):
    # expected with HFA
    expected_home = expected_home_win_prob(home_elo, away_elo, hfa=hfa)

    # cap margin (for sims we typically keep margin=1 unless you simulate scorelines)
    margin = max(1, min(int(margin), int(cap_margin)))

    # dynamic k adjustment for draws (same style as your model)
    if result_home == 0.5:
        surprise_factor = abs(result_home - expected_home)
        k_adjust = 0.5 + (upset_multiplier * surprise_factor)
    else:
        k_adjust = 1.0

    change_home = float(k_adjust * k * margin * (result_home - expected_home))
    return home_elo + change_home, away_elo - change_home


def run_elo_over_matches(matches: pd.DataFrame, base_elo=BASE_ELO) -> dict[str, float]:
    """
    Run Elo over matches in the order they appear (assumes already sorted).
    Uses margin=1 for soccer unless you later add scoreline simulation.
    """
    team_elos = defaultdict(lambda: float(base_elo))

    for _, row in matches.iterrows():
        home = row["Home Team"]
        away = row["Away Team"]
        hs = int(row["Home Score"])
        as_ = int(row["Away Score"])

        result_home = 1.0 if hs > as_ else 0.0 if hs < as_ else 0.5

        home_elo = float(team_elos[home])
        away_elo = float(team_elos[away])

        new_home, new_away = elo_update(home_elo, away_elo, result_home, margin=1)
        team_elos[home] = new_home
        team_elos[away] = new_away

    return dict(team_elos)


# =========================
# DATA PREP
# =========================
def load_and_prepare(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # column harmonization
    if "DateRaw" not in df.columns:
        if "Date" in df.columns:
            df["DateRaw"] = df["Date"]
        else:
            raise ValueError("Expected DateRaw (or Date) column in CSV.")

    if "TimeRaw" not in df.columns:
        if "Time" in df.columns:
            df["TimeRaw"] = df["Time"]
        else:
            df["TimeRaw"] = ""

    # normalize teams
    df["Home Team"] = df["Home Team"].apply(normalize_team_name)
    df["Away Team"] = df["Away Team"].apply(normalize_team_name)

    # allowlist filter (Metro-only)
    hk = df["Home Team"].map(canonicalize)
    ak = df["Away Team"].map(canonicalize)
    df = df[hk.isin(METRO_TEAMS) & ak.isin(METRO_TEAMS)].copy()

    # drop excluded game types if present
    if "Game Type" in df.columns:
        df["Game Type"] = df["Game Type"].fillna("").astype(str)
        df = df[~df["Game Type"].isin(EXCLUDE_GAME_TYPES)].copy()

    # scores
    df["Home Score"] = pd.to_numeric(df["Home Score"], errors="coerce")
    df["Away Score"] = pd.to_numeric(df["Away Score"], errors="coerce")

    # stable tie-breaker
    if "RowIndex" not in df.columns:
        df["RowIndex"] = np.arange(len(df), dtype=int)

    # datetime for ordering
    df["MatchDateTime"] = [
        parse_match_datetime(s, d, t)
        for s, d, t in zip(df["Season"], df["DateRaw"], df["TimeRaw"])
    ]
    df["DateISO"] = pd.to_datetime(df["MatchDateTime"], errors="coerce").dt.strftime("%Y-%m-%d")

    # sort season chronologically, stable tie-breaker
    df["MatchDateTimeFill"] = df["MatchDateTime"].fillna(pd.Timestamp.max)
    df = df.sort_values(["Season", "MatchDateTimeFill", "RowIndex"], kind="mergesort").reset_index(drop=True)
    df = df.drop(columns=["MatchDateTimeFill"])

    return df


def estimate_draw_rate_from_history(df: pd.DataFrame, season_cutoff: str) -> float:
    hist = df[(df["Season"] < season_cutoff) & df["Home Score"].notna() & df["Away Score"].notna()].copy()
    if hist.empty:
        return 0.18
    ties = (hist["Home Score"] == hist["Away Score"]).mean()
    # clamp to reasonable soccer range
    return float(np.clip(ties, 0.08, 0.35))


def compute_current_state_for_season(df: pd.DataFrame, season: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    season_df = df[df["Season"] == season].copy()
    played = season_df[season_df["Home Score"].notna() & season_df["Away Score"].notna()].copy()
    unplayed = season_df[season_df["Home Score"].isna() | season_df["Away Score"].isna()].copy()
    return played, unplayed


def points_from_played(played_matches: pd.DataFrame, teams: list[str]) -> dict[str, int]:
    pts = {t: 0 for t in teams}
    for _, r in played_matches.iterrows():
        home, away = r["Home Team"], r["Away Team"]
        hs, a = int(r["Home Score"]), int(r["Away Score"])
        if hs > a:
            pts[home] += 3
        elif hs < a:
            pts[away] += 3
        else:
            pts[home] += 1
            pts[away] += 1
    return pts


# =========================
# MONTE CARLO
# =========================
def simulate_season(
    fixtures: pd.DataFrame,
    starting_elos: dict[str, float],
    base_points: dict[str, int],
    n_sims: int,
    draw_rate: float,
    seed: int = 42,
    update_elo_during_sim: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      seed_probs_df: P(team finishes at seed k)
      points_summary_df: mean/p10/p50/p90 points per team
    """
    rng = np.random.default_rng(seed)

    teams = sorted(set(fixtures["Home Team"]).union(set(fixtures["Away Team"])))
    n_teams = len(teams)
    team_to_idx = {t: i for i, t in enumerate(teams)}

    # Pre-extract fixtures arrays for speed
    home_arr = fixtures["Home Team"].to_numpy()
    away_arr = fixtures["Away Team"].to_numpy()

    # Storage
    seed_counts = np.zeros((n_teams, n_teams), dtype=int)  # [team, seed-1]
    points_matrix = np.zeros((n_sims, n_teams), dtype=int)

    # For tie-breaks, we’ll use: Points -> Wins -> Final Elo -> Random
    # We track wins too:
    wins_matrix = np.zeros((n_sims, n_teams), dtype=int)

    for sim in range(n_sims):
        # init
        elos = dict(starting_elos)
        pts = {t: int(base_points.get(t, 0)) for t in teams}
        wins = {t: 0 for t in teams}

        # play fixtures in order
        for h, a in zip(home_arr, away_arr):
            he = float(elos.get(h, BASE_ELO))
            ae = float(elos.get(a, BASE_ELO))
            p_home_win = expected_home_win_prob(he, ae, hfa=HFA)

            # incorporate draw rate
            p_draw = draw_rate
            p_home = (1.0 - p_draw) * p_home_win
            p_away = (1.0 - p_draw) * (1.0 - p_home_win)

            u = rng.random()
            if u < p_away:
                # away win
                pts[a] += 3
                wins[a] += 1
                result_home = 0.0
            elif u < p_away + p_draw:
                # draw
                pts[h] += 1
                pts[a] += 1
                result_home = 0.5
            else:
                # home win
                pts[h] += 3
                wins[h] += 1
                result_home = 1.0

            if update_elo_during_sim:
                new_he, new_ae = elo_update(he, ae, result_home, margin=1)
                elos[h] = new_he
                elos[a] = new_ae

        # Build ranking with tie-breakers: points, wins, final elo, random
        # random is simulated by adding tiny noise
        elo_vec = np.array([elos.get(t, BASE_ELO) for t in teams], dtype=float)
        pts_vec = np.array([pts[t] for t in teams], dtype=int)
        wins_vec = np.array([wins[t] for t in teams], dtype=int)
        noise = rng.random(n_teams) * 1e-6

        # sort descending by pts, wins, elo
        order = np.lexsort((noise, -elo_vec, -wins_vec, -pts_vec))
        # lexsort sorts ascending; we already negated; order is correct

        # record seeds + points
        inv_rank = np.empty(n_teams, dtype=int)
        inv_rank[order] = np.arange(1, n_teams + 1)  # seed numbers
        for i, t in enumerate(teams):
            seed_counts[i, inv_rank[i] - 1] += 1

        points_matrix[sim, :] = pts_vec
        wins_matrix[sim, :] = wins_vec

    # Seed probabilities
    seed_probs = seed_counts / n_sims
    seed_probs_df = pd.DataFrame(
        seed_probs,
        index=teams,
        columns=[f"Seed {k}" for k in range(1, n_teams + 1)],
    ).reset_index().rename(columns={"index": "Team"})

    # Points summary
    points_summary = []
    for i, t in enumerate(teams):
        arr = points_matrix[:, i]
        points_summary.append({
            "Team": t,
            "Mean Points": float(arr.mean()),
            "P10 Points": float(np.percentile(arr, 10)),
            "P50 Points": float(np.percentile(arr, 50)),
            "P90 Points": float(np.percentile(arr, 90)),
        })
    points_summary_df = pd.DataFrame(points_summary).sort_values("Mean Points", ascending=False).reset_index(drop=True)

    return seed_probs_df, points_summary_df


# =========================
# RUN WORKFLOW
# =========================
df_all = load_and_prepare(INPUT_CSV)

# Teams we care about (your explicit list), and ensure we only simulate those
TEAMS_2526 = [normalize_team_name(t) for t in METRO_TEAMS_RAW]
teams_set = {canonicalize(t) for t in TEAMS_2526}

df_all = df_all[
    df_all["Home Team"].map(canonicalize).isin(teams_set) &
    df_all["Away Team"].map(canonicalize).isin(teams_set)
].copy()

# Historical matches (train Elo up to the start of 2025-26)
history_played = df_all[(df_all["Season"] < SEASON_TO_SIMULATE) & df_all["Home Score"].notna() & df_all["Away Score"].notna()].copy()

# Estimate draw rate if not provided
draw_rate = float(DRAW_RATE) if DRAW_RATE is not None else estimate_draw_rate_from_history(df_all, SEASON_TO_SIMULATE)

# Elo baseline from history
starting_elos = run_elo_over_matches(history_played, base_elo=BASE_ELO)

# Get 2025-26 played/unplayed
played_2526, fixtures_2526 = compute_current_state_for_season(df_all, SEASON_TO_SIMULATE)

# If some 2025-26 games already have results, incorporate them into current Elo + current points (then simulate remaining)
base_points = {t: 0 for t in TEAMS_2526}
if not played_2526.empty:
    # Update Elo through already played 2025-26 games (in chronological order)
    # (Ensures you simulate from “today” state rather than preseason state)
    temp = played_2526.copy()
    temp = temp.sort_values(["MatchDateTime", "RowIndex"], kind="mergesort")
    updated_elos = dict(starting_elos)
    # replay Elo updates on top of history baseline
    # (use the same core update function)
    for _, r in temp.iterrows():
        h, a = r["Home Team"], r["Away Team"]
        he = float(updated_elos.get(h, BASE_ELO))
        ae = float(updated_elos.get(a, BASE_ELO))
        hs, as_ = int(r["Home Score"]), int(r["Away Score"])
        res_home = 1.0 if hs > as_ else 0.0 if hs < as_ else 0.5
        new_he, new_ae = elo_update(he, ae, res_home, margin=1)
        updated_elos[h] = new_he
        updated_elos[a] = new_ae

    starting_elos = updated_elos
    base_points = points_from_played(temp, TEAMS_2526)

# Fixtures to simulate: unplayed 2025-26 only, in chronological order
fixtures_2526 = fixtures_2526.copy()
fixtures_2526 = fixtures_2526.sort_values(["MatchDateTime", "RowIndex"], kind="mergesort")

# Sanity: everyone plays each other once? (Optional check)
# You can uncomment to verify:
# pairs = set()
# dupes = 0
# for _, r in fixtures_2526.iterrows():
#     p = tuple(sorted([r["Home Team"], r["Away Team"]]))
#     if p in pairs: dupes += 1
#     pairs.add(p)
# print("Unique pairs:", len(pairs), "Dupes:", dupes)

seed_probs_df, points_summary_df = simulate_season(
    fixtures=fixtures_2526[["Home Team", "Away Team", "MatchDateTime", "RowIndex"]],
    starting_elos=starting_elos,
    base_points=base_points,
    n_sims=N_SIMS,
    draw_rate=draw_rate,
    seed=SEED,
    update_elo_during_sim=UPDATE_ELO_DURING_SIM,
)

seed_out = OUTPUT_DIR / f"metroleague_{SEASON_TO_SIMULATE}_sim_seed_probabilities.csv"
pts_out = OUTPUT_DIR / f"metroleague_{SEASON_TO_SIMULATE}_sim_points_summary.csv"
settings_out = OUTPUT_DIR / f"metroleague_{SEASON_TO_SIMULATE}_sim_settings.txt"

seed_probs_df.to_csv(seed_out, index=False, float_format="%.4f")
points_summary_df.to_csv(pts_out, index=False, float_format="%.2f")

settings_out.write_text(
    "\n".join([
        f"SEASON_TO_SIMULATE={SEASON_TO_SIMULATE}",
        f"N_SIMS={N_SIMS}",
        f"SEED={SEED}",
        f"BASE_ELO={BASE_ELO}",
        f"K={K}",
        f"HFA={HFA}",
        f"CAP_MARGIN={CAP_MARGIN}",
        f"UPSET_MULTIPLIER={UPSET_MULTIPLIER}",
        f"UPDATE_ELO_DURING_SIM={UPDATE_ELO_DURING_SIM}",
        f"DRAW_RATE_USED={draw_rate:.4f}",
        f"FIXTURES_SIMULATED={len(fixtures_2526)}",
        f"PLAYED_ALREADY_IN_SEASON={len(played_2526)}",
    ]),
    encoding="utf-8"
)

print("Monte Carlo complete.")
print("Saved:")
print(" -", seed_out)
print(" -", pts_out)
print(" -", settings_out)

Monte Carlo complete.
Saved:
 - C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\Monte Carlo Sim Output\2.24\metroleague_2025-26_sim_seed_probabilities.csv
 - C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\Monte Carlo Sim Output\2.24\metroleague_2025-26_sim_points_summary.csv
 - C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\Monte Carlo Sim Output\2.24\metroleague_2025-26_sim_settings.txt
